# Árbol de decisión: Restaurante(<20)

**Integrante:** [nombre]. **Laboratorio 3 — Ruta Alpes.**

Seguimos la estructura de la [práctica del curso](../MaterialDeClase-ISIS-2611/202620/Prácticas/P8_Árboles de decisión/ArbolesDecision.ipynb).
La limpieza se documenta en el notebook compartido. Este archivo parte del CSV preparado por el equipo.
Plantilla sin ejecutar: completar las decisiones y los análisis pendientes con evidencia propia.

**Pregunta inicial:** ¿qué características del conductor y del trayecto ayudan a predecir la aceptación de este cupón?


## 1. Importación de librerías


In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_score, recall_score, f1_score)


In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text


## 2. Carga de los datos

Ejecutar desde la raíz de LABORATORIO-3-AM. El CSV debe conservar id_fila, cupon y acepta_cupon en Sí/No.
Puede contener faltantes en predictores: se imputan dentro del pipeline. Los faltantes de la etiqueta no se imputan.


In [ ]:
CUPON = 'Restaurante(<20)'
ALGORITMO = 'arbol_decision'
SEMILLA = 42
N_FOLDS = 5
N_JOBS = -1
RUTA_DATOS = Path('data/procesados/datos_limpios.csv')
RUTA_RESULTADOS = Path('resultados') / 'restaurante_arbol_decision'
# Completar con las decisiones de la limpieza; mantenerlas entre algoritmos.
# vehiculo, tiempo_cupon_mayor_5min y direccion_opuesta
# ya se eliminaron en el notebook de limpieza compartido.
COLUMNAS_EXCLUIR = []
COLUMNAS_NUMERICAS = ['temperatura']

if not RUTA_DATOS.exists():
    raise FileNotFoundError('Falta data/procesados/datos_limpios.csv. Preparar el CSV y abrir desde la raíz del laboratorio.')
datos = pd.read_csv(RUTA_DATOS)


In [ ]:
requeridas = {'id_fila', 'cupon', 'acepta_cupon'}
if not requeridas.issubset(datos.columns):
    raise ValueError(f'Faltan columnas: {requeridas - set(datos.columns)}')
if datos['id_fila'].isna().any() or not datos['id_fila'].is_unique:
    raise ValueError('id_fila debe ser único y no tener faltantes.')
data = datos.loc[datos['cupon'].eq(CUPON)].copy()
data = data.sort_values('id_fila').set_index('id_fila')
if data.empty:
    raise ValueError(f'No se encontraron filas de {CUPON}. Revisar las categorías del CSV.')
if data['acepta_cupon'].isna().any() or not set(data['acepta_cupon']).issubset({'Sí', 'No'}):
    raise ValueError('acepta_cupon debe contener solamente Sí y No, sin faltantes.')
data.head()


## 3. Revisión del subconjunto

La exploración completa y las correcciones pertenecen al notebook de limpieza. Aquí comprobamos qué datos llegan al modelo.


In [ ]:
data.shape


In [ ]:
data.isna().sum()


In [ ]:
data['acepta_cupon'].value_counts()


**Por completar:** resumir la distribución de clases, las columnas descartadas y las decisiones de limpieza relevantes para este cupón. No suponer que el balanceo mejorará el resultado antes de medirlo.


## 4. Partición de los datos

Sí se representa con 1 y No con 0. Excluimos cupon, que es constante, y la etiqueta de los predictores. id_fila quedó como índice.
La semilla, la fracción de test y los folds deben ser iguales para los tres modelos del mismo cupón.


In [ ]:
X = data.drop(columns=['acepta_cupon', 'cupon', *COLUMNAS_EXCLUIR])
y = data['acepta_cupon'].map({'No': 0, 'Sí': 1}).astype(int)
numericas_features = [c for c in COLUMNAS_NUMERICAS if c in X.columns]
categoricas_features = [c for c in X.columns if c not in numericas_features]

# Conservamos los faltantes como np.nan para SimpleImputer.
# Edad, ingreso y frecuencias son categorías según el diccionario.
for columna in categoricas_features:
    X[columna] = X[columna].astype('string').astype(object)
    X[columna] = X[columna].where(X[columna].notna(), np.nan)
for columna in numericas_features:
    X[columna] = pd.to_numeric(X[columna], errors='raise')
    if not np.isfinite(X[columna].dropna().to_numpy()).all():
        raise ValueError(f'Hay valores no finitos en {columna}. Revisar la limpieza.')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEMILLA, stratify=y
)
if y_train.nunique() != 2 or y_train.value_counts().min() < N_FOLDS:
    raise ValueError('Se requieren ambas clases y suficientes ejemplos para cinco folds.')
if numericas_features and X_train[numericas_features].isna().all().any():
    raise ValueError('Hay columnas numéricas completamente vacías en train. Revisar si deben excluirse.')
clase_minoritaria = int(y_train.value_counts().idxmin())
kfold = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEMILLA)
print('Train:', X_train.shape, '| Test:', X_test.shape)
print('Clase minoritaria en train:', {0: 'No', 1: 'Sí'}[clase_minoritaria])


## 5. Construccion del pipeline

Seguimos el esquema de SimpleImputer y ColumnTransformer de la practica 7. La mediana numerica y la moda de cada variable categorica se calculan dentro de cada fold de entrenamiento.


Aunque los árboles no necesitan escalamiento, lo mantenemos en las dos variantes de Restaurante porque SMOTE calcula distancias. Así el experimento solo cambia el muestreo.


In [ ]:
numerica_transformer = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])


Codificamos las categorías con OneHotEncoder(handle_unknown='ignore'), como en la práctica de árboles.


In [ ]:
categorica_transformer = Pipeline(steps=[
    ('imputador', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')),
])


In [ ]:
preprocesador = ColumnTransformer(transformers=[
    ('num', numerica_transformer, numericas_features),
    ('cat', categorica_transformer, categoricas_features),
])


In [ ]:
decision_tree_pipeline = Pipeline(steps=[
    ('preprocesador', preprocesador),
    ('model', DecisionTreeClassifier(random_state=SEMILLA)),
])


## 6. Búsqueda de hiperparámetros

Usamos GridSearchCV de forma explícita, como en las prácticas. Se selecciona por F1 macro en CV, dando igual peso a ambas clases. La práctica 7 usa F1 binario; unificamos F1 macro con las prácticas 8–10 para comparar los tres algoritmos. También reportamos el F1 de aceptación.


Retomamos el criterio y la profundidad de la práctica 8. Añadimos min_samples_leaf, que el enunciado exige explorar. No es necesario copiar todos los hiperparámetros de la práctica: esta grilla se concentra en los requisitos del laboratorio.


In [ ]:
param_grid = {
    'model__criterion': ['gini', 'entropy', 'log_loss'],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_leaf': [1, 3, 5],
}


In [ ]:
decision_tree_grid = GridSearchCV(
    estimator=decision_tree_pipeline, param_grid=param_grid,
    cv=kfold, scoring='f1_macro', n_jobs=N_JOBS,
    return_train_score=True, error_score='raise',
)
decision_tree_grid.fit(X_train, y_train)


In [ ]:
print('Mejores hiperparámetros:', decision_tree_grid.best_params_)
print('F1 macro promedio en CV:', decision_tree_grid.best_score_)
resultados_cv = pd.DataFrame(decision_tree_grid.cv_results_)
columnas_cv = [c for c in resultados_cv.columns if c.startswith('param_')]
columnas_cv += ['mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score']
display(resultados_cv[columnas_cv].sort_values('rank_test_score').head(10))
mejor_modelo = decision_tree_grid.best_estimator_
variante_elegida = 'sin_balanceo'


**Por completar:** explicar los hiperparámetros ganadores, comparar train con validación y revisar la dispersión entre folds. Si aparecen advertencias de convergencia, resolverlas antes de reportar el resultado.


## 7. Comparación con SMOTE

Adaptamos el esquema ImbPipeline → SMOTE → modelo de la [práctica 9](../MaterialDeClase-ISIS-2611/202620/Prácticas/P9_KNN_SMOTE/P9_KNN_SMOTE.ipynb). La grilla, los folds y el preprocesamiento son los mismos en ambas variantes. Solo se remuestrean las filas de entrenamiento de cada fold.

**Limitación que se debe discutir:** la práctica usa variables numéricas. Aquí SMOTE recibe también categorías codificadas y puede generar valores one-hot fraccionarios que no corresponden a categorías reales. Este experimento sigue el método visto en clase, pero esa limitación debe constar en la recomendación. SMOTENC o sobremuestreo aleatorio serían alternativas para contrastar si el equipo amplía el estudio.


In [ ]:
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# SMOTE usa cinco vecinos: cada fold necesita al menos seis ejemplos minoritarios.
for indices_train, _ in kfold.split(X_train, y_train):
    if y_train.iloc[indices_train].value_counts().min() <= 5:
        raise ValueError('No hay suficientes ejemplos minoritarios por fold para SMOTE(k_neighbors=5).')


In [ ]:
smote_pipeline = ImbPipeline(steps=[
    ('preprocesador', clone(preprocesador)),
    ('smote', SMOTE(random_state=SEMILLA)),
    ('model', clone(decision_tree_pipeline.named_steps['model'])),
])
smote_grid = GridSearchCV(
    estimator=smote_pipeline, param_grid=param_grid,
    cv=kfold, scoring='f1_macro', n_jobs=N_JOBS,
    return_train_score=True, error_score='raise',
)
smote_grid.fit(X_train, y_train)
print('Mejores parámetros con SMOTE:', smote_grid.best_params_)
print('F1 macro en CV con SMOTE:', smote_grid.best_score_)


In [ ]:
comparacion_cv = pd.DataFrame({
    'variante': ['sin_balanceo', 'smote'],
    'f1_macro_cv': [decision_tree_grid.best_score_, smote_grid.best_score_],
})
display(comparacion_cv)
# Decidimos antes de mirar test. En empate conservamos sin balanceo.
if smote_grid.best_score_ > decision_tree_grid.best_score_:
    mejor_modelo = smote_grid.best_estimator_
    variante_elegida = 'smote'
print('Variante elegida por CV:', variante_elegida)


## 8. Evaluación

Como en las prácticas, revisamos reporte de clasificación y matriz de confusión. Test conserva su distribución original y no se usa para ajustar parámetros ni escoger la variante.


In [ ]:
y_pred = mejor_modelo.predict(X_test)
print(classification_report(y_test, y_pred, labels=[0, 1], target_names=['No', 'Sí'], zero_division=0))
print(confusion_matrix(y_test, y_pred, labels=[0, 1]))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=[0, 1], display_labels=['No', 'Sí'], cmap='Blues'
)
plt.title(f'{CUPON} — {ALGORITMO} — {variante_elegida}')
plt.show()


In [ ]:
def metricas_modelo(y_true, y_pred):
    return {
        'precision_si': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_si': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_si': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'recall_minoritaria': recall_score(y_true, y_pred, pos_label=clase_minoritaria, zero_division=0),
    }

comparacion_train_test = pd.DataFrame({
    'train': metricas_modelo(y_train, mejor_modelo.predict(X_train)),
    'test': metricas_modelo(y_test, y_pred),
}).T
display(comparacion_train_test.round(3))


In [ ]:
pred_sin_balanceo = decision_tree_grid.best_estimator_.predict(X_test)
resultados = pd.DataFrame([{
    'cupon': CUPON, 'algoritmo': ALGORITMO, 'variante': 'sin_balanceo',
    'elegida_por_cv': variante_elegida == 'sin_balanceo',
    'f1_macro_cv': decision_tree_grid.best_score_, 'clase_minoritaria': clase_minoritaria,
    **metricas_modelo(y_test, pred_sin_balanceo),
}])


In [ ]:
pred_smote = smote_grid.best_estimator_.predict(X_test)
fila_smote = pd.DataFrame([{
    'cupon': CUPON, 'algoritmo': ALGORITMO, 'variante': 'smote',
    'elegida_por_cv': variante_elegida == 'smote',
    'f1_macro_cv': smote_grid.best_score_, 'clase_minoritaria': clase_minoritaria,
    **metricas_modelo(y_test, pred_smote),
}])
resultados = pd.concat([resultados, fila_smote], ignore_index=True)


In [ ]:
display(resultados.round(3))


In [ ]:
resultados.set_index('variante')[['f1_si', 'f1_macro', 'recall_minoritaria']].plot.bar(
    ylim=(0, 1), figsize=(8, 4), title='Sin balanceo frente a SMOTE'
)
plt.xticks(rotation=0)
plt.show()


**Por completar:** interpretar errores y brecha train–test. En Restaurante, comparar F1 de aceptación, F1 macro y recall minoritario y justificar si conviene SMOTE. En los datos originales la minoritaria de Restaurante es No; verificar después de limpiar. No cambiar la selección en función de test.


## 9. Interpretación del modelo

Seguimos el enfoque de la práctica: coeficientes para regresión, y reducción de impureza y reglas para árboles/bosques. Describen asociaciones del modelo, no relaciones causales.


In [ ]:
preprocesador_fit = mejor_modelo.named_steps['preprocesador']
modelo_fit = mejor_modelo.named_steps['model']
feature_names = preprocesador_fit.get_feature_names_out()


In [ ]:
importancias = pd.Series(modelo_fit.feature_importances_, index=feature_names)
importancias = importancias.sort_values(ascending=False)
display(importancias.head(20))
importancias.head(15).sort_values().plot.barh(figsize=(8, 5))
plt.xlabel('Importancia por reducción de impureza')
plt.title(f'Variables relevantes — {CUPON}')
plt.show()


Las importancias corresponden a columnas codificadas. Una variable original puede aparecer en varias categorías. La reducción de impureza puede favorecer variables con muchas categorías; considerar la correlación y evitar conclusiones causales. **Por completar:** identificar variables originales relevantes e indicar qué categorías aparecen.


In [ ]:
plt.figure(figsize=(20, 10))
plot_tree(modelo_fit, feature_names=list(feature_names), class_names=['No', 'Sí'],
          max_depth=3, filled=True, rounded=True, fontsize=8)
plt.title('Primeros niveles del árbol; vista truncada')
plt.show()
print(export_text(modelo_fit, feature_names=list(feature_names), max_depth=4))


**Por completar:** traducir dos o tres reglas a lenguaje de negocio y explicar utilidad y limitación. Si se seleccionó SMOTE, los conteos incluyen ejemplos sintéticos y no representan la distribución original.


## 10. Guardar resultados

Persistimos el pipeline, como en la práctica 7, y las métricas para integrar después. Este modelo se ajustó solo en train. El ganador de cada cupón se reajustará con todos sus datos etiquetados al generar el CSV externo.


In [ ]:
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
resultados.to_csv(RUTA_RESULTADOS / 'metricas_test.csv', index=False, encoding='utf-8-sig')
resultados_cv.to_csv(RUTA_RESULTADOS / 'cv_sin_balanceo.csv', index=False, encoding='utf-8-sig')
particion = pd.concat([
    pd.DataFrame({'id_fila': X_train.index, 'conjunto': 'train'}),
    pd.DataFrame({'id_fila': X_test.index, 'conjunto': 'test'}),
], ignore_index=True)
particion.to_csv(RUTA_RESULTADOS / 'particion.csv', index=False)
pd.DataFrame({'id_fila': X_test.index, 'real': y_test.to_numpy(), 'prediccion': y_pred}).to_csv(
    RUTA_RESULTADOS / 'predicciones_test_interno.csv', index=False
)
joblib.dump(mejor_modelo, RUTA_RESULTADOS / 'pipeline_evaluado.joblib')


In [ ]:
pd.DataFrame(smote_grid.cv_results_).to_csv(
    RUTA_RESULTADOS / 'cv_smote.csv', index=False, encoding='utf-8-sig'
)


In [ ]:
importancias.rename('importancia').to_csv(RUTA_RESULTADOS / 'importancias.csv', encoding='utf-8-sig')


## 11. Análisis para el informe conjunto

Completar después de ejecutar y contrastar los seis modelos:

- ¿Qué características se relacionan con la aceptación? ¿Cuáles parecen redundantes?
- ¿Cómo cambia el desempeño entre algoritmos y cupones? ¿Qué explicaría las diferencias?
- ¿Conviene el balanceo en Restaurante considerando F1 y recall minoritario?
- ¿Qué utilidad y limitación tienen las reglas o las importancias?
- ¿Cómo se integraría en la aplicación de conducción?
- ¿Qué limitaciones, fuentes de sesgo e implicaciones tiene usar edad, género, estado civil o ingreso? ¿Qué cambiaría en precios dinámicos?

El documento final debe responder las ocho preguntas del enunciado explícitamente, con evidencia propia.


## 12. Uso de herramientas de IA generativa

**Herramienta y propósito:** Codex (OpenAI), apoyo conceptual y generación/adaptación inicial de código a las prácticas 7–10 del curso. La plantilla se entrega sin resultados de entrenamiento del laboratorio.

**Prompts principales resumidos:**

1. Revisar el lab 3 de 202620, repartir los seis modelos entre dos integrantes y crear un notebook por modelo; dejar la limpieza al estudiante.
2. Revisar las prácticas y adaptar los notebooks a los métodos y la forma de trabajo del curso.

**Evaluación crítica del estudiante — completar al menos dos:** ¿qué fue útil y cómo se verificó?, ¿qué errores o limitaciones se identificaron?, ¿qué decisiones se cambiaron y por qué?, ¿qué conceptos del curso permitieron revisar la propuesta?

**Aportes propios — completar:** decisiones de limpieza, cambios al código, justificaciones, ejecución, interpretación y aprendizajes. Registrar los prompts adicionales que influyan en la entrega.
